# V63 - Full build from hunt winners (Kaggle Commit)

This is the full build the hunt was for. It composes every surviving winner at heavy scale: word-only chat, decoded assessment, mentions, and content cosine as views; per-module LightGBM-1200 plus Ridge heads; multilingual-E5 dense view; calibrated view combination by grouped OOF weights; and one champion-relative action (no-chat assessment override) behind slice, safety, and geometry locks. Excluded with reason: bigrams (lose), residuals (noise), CatBoostRanker (constant on thin signal), Qwen3-8B (16GB VRAM, T4-borderline, hours for an unproven slice). SMOKE=1 runs a tiny local verification. Commit needs internet ON for the E5 weights and the champion file as an input dataset.

In [ ]:
import json
import os
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = bool(int(os.getenv("V63_SMOKE", "0")))
SEED = 111
N_SPLITS = 2 if SMOKE else 5
N_EST = 20 if SMOKE else 1200
GATE_FLOOR = 0.002
SLICE_FLOOR = 0.01
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "full build composes onto the champion file"
USE_E5 = not SMOKE
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v63")
OUT.mkdir(parents=True, exist_ok=True)
print(f"SMOKE={SMOKE} DATA={DATA.resolve()} USE_E5={USE_E5}")

## Views at full scale

Word-only chat, decoded assessment, mention counts, training-free content cosine, and E5 dense (skipped in smoke). One Ridge matrix per view is kept for the combination stage.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())
MN = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", (str(row["module_name"]) + " " + str(row["description_and_syllabus"])).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        MN[i, j] = sum(d.count(k) for k in keys)
MN = (MN - MN[:n_train].mean(0)) / (MN[:n_train].std(0) + 1e-9)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(docs + catalog["description_and_syllabus"].fillna("").tolist())
syll = [catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "" for m in MODS]
CM = cosine_similarity(vec.transform(docs), vec.transform(syll))
VIEWS = {"word": W, "assess": A, "mentions": np.nan_to_num(MN), "content": CM}
if USE_E5:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("intfloat/multilingual-e5-base")
    E = enc.encode(["query: " + d for d in docs], batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    VIEWS["e5"] = np.asarray(E)
    print("views:", {k: v.shape for k, v in VIEWS.items()})

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def asdense(M, idx):
    m = M[idx] if not hasattr(M, "toarray") else M[idx].toarray()
    return np.asarray(m, dtype=np.float32)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

## Heavy heads, OOF weights, no-chat action

Per-module LightGBM plus Ridge per view under grouped folds. View weights come from grouped OOF, never from test. The no-chat override acts on the champion file only if its slice lock, safety lock, and geometry lock all pass.

In [ ]:
import lightgbm as lgb
names = list(VIEWS)
nV = len(names)
oof = {k: np.zeros_like(Y) for k in names}
pred = {k: np.zeros((len(test_ids), len(MODS))) for k in names}
fids = np.zeros(n_train, int)
gbm_te = np.zeros((len(test_ids), len(MODS)))
outer = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(outer.split(asdense(VIEWS[names[0]], range(n_train)), strata, fold_groups)):
    for k in names:
        Xtr, Xva = asdense(VIEWS[k][:n_train], tr), asdense(VIEWS[k][:n_train], va)
        Xte = asdense(VIEWS[k][n_train:], range(len(test_ids)))
        for j in range(len(MODS)):
            r = Ridge(alpha=1.0)
            r.fit(Xtr, Y[tr, j])
            oof[k][va, j] = r.predict(Xva)
            pred[k][:, j] += r.predict(Xte) / N_SPLITS
            if k == "word":
                g = lgb.LGBMRegressor(objective="regression", n_estimators=N_EST, learning_rate=0.05,
                                        num_leaves=31, min_data_in_leaf=50, verbose=-1, deterministic=True, seed=SEED)
                g.fit(Xtr, Y[tr, j])
                oof[k][va, j] = (oof[k][va, j] + g.predict(Xva)) / 2
                gbm_te[:, j] += g.predict(Xte) / N_SPLITS
    fids[va] = fi + 1
    print(f"outer {fi + 1} done", flush=True)
pred["word"] = (pred["word"] + gbm_te) / 2
for k in names:
    print(f"{k}: OOF={ndcg5(Y, oof[k]).mean():.5f}")
w = np.array([max(ndcg5(Y, oof[k]).mean(), 0) for k in names])
w = w / w.sum()
combo = sum(w[i] * oof[k] for i, k in enumerate(names))
combo_te = sum(w[i] * pred[k] for i, k in enumerate(names))
print("weights:", dict(zip(names, np.round(w, 3))), f"combo OOF={ndcg5(Y, combo).mean():.5f}")

is_nochat = np.array([u not in by_user for u in all_users])
nochat_tr = is_nochat[:n_train]
base = ndcg5(Y, combo)
act = combo.copy()
act[nochat_tr] = oof["assess"][nochat_tr]
d = ndcg5(Y, act) - base
ds = d[nochat_tr]
lo_s, hi_s = np.quantile([rng.choice(ds, len(ds), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], act[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"no-chat action: pooled={d.mean():+.5f} slice CI95=[{lo_s:+.5f},{hi_s:+.5f}] worst-fold={wf:+.5f}")

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
S = C.copy()
full_assess = pred["assess"]
S[is_nochat[n_train:]] = full_assess[is_nochat[n_train:]]
changed = set(np.where((S != C).any(1))[0].tolist())
allowed = set(np.where(is_nochat[n_train:])[0].tolist())
adopt = (lo_s > SLICE_FLOOR) and (wf >= 0) and (changed <= allowed)
print("ADOPT no-chat action onto champion" if adopt else "FAIL-CLOSE: champion untouched")
final = S if adopt else C
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v63_primary.csv", index=False)
print("wrote", OUT / "submission_v63_primary.csv", "sha", sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16])